Импорты, seed и устройство

In [1]:
import os
import random
import torch
import torchvision
import numpy as np

# Создаем структуру папок (согласно заданию)
os.makedirs("artifacts/figures", exist_ok=True)

# Фиксируем seed для жесткого контроля над воспроизводимостью
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Проверка захвата вычислительных мощностей
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


In [4]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Смена вектора: загружаем EMNIST (Вариант B)
train_dataset_full = torchvision.datasets.EMNIST(root='./data', split='balanced', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.EMNIST(root='./data', split='balanced', train=False, download=True, transform=transform)

# Изоляция валидационной выборки (80/20)
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(
    train_dataset_full, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for X, y in train_loader:
    print(f"Форма тензора X: {X.shape}")
    print(f"Форма тензора меток y: {y.shape}")
    print(f"Диапазон X: [{X.min():.2f}, {X.max():.2f}]")
    break

100.0%


Форма тензора X: torch.Size([64, 1, 28, 28])
Форма тензора меток y: torch.Size([64])
Диапазон X: [-1.00, 1.00]


In [6]:
import copy
import torch.nn as nn
import torch.optim as optim

class MLP(nn.Module):
    def __init__(self, use_dropout=False, use_bn=False, drop_p=0.3):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = []
        
        # Скрытый слой 1
        layers.append(nn.Linear(28 * 28, 256))
        if use_bn: layers.append(nn.BatchNorm1d(256))
        layers.append(nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(drop_p))
            
        # Скрытый слой 2
        layers.append(nn.Linear(256, 128))
        if use_bn: layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(drop_p))
            
        # Выходной слой (47 классов для EMNIST balanced)
        layers.append(nn.Linear(128, 47))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(self.flatten(x))

def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train() # Перевод сети в режим активной адаптации
    running_loss, correct, total = 0.0, 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        _, predicted = outputs.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
        
    return running_loss / total, correct / total

def evaluate(model, dataloader, criterion):
    model.eval() # Блокировка механизмов регуляризации для чистой проверки
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad(): # Отключение градиентов
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            running_loss += loss.item() * X.size(0)
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
            
    return running_loss / total, correct / total

In [7]:
import json

results = []
histories = {}

def run_experiment(exp_id, model_kwargs, opt_name, lr, momentum=0, weight_decay=0, epochs=10, patience=None):
    set_seed(42) # Жесткий контроль начальных условий
    model = MLP(**model_kwargs).to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Выбор инструмента оптимизации
    if opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
        
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_val_loss = float('inf')
    epochs_trained = 0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        v_loss, v_acc = evaluate(model, val_loader, criterion)
        
        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        epochs_trained += 1
        
        # Фиксация лучшего состояния
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_val_loss = v_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        # Механизм ранней остановки (EarlyStopping)
        if patience and patience_counter >= patience:
            print(f"[{exp_id}] Обучение принудительно прервано на эпохе {epoch+1}. Модель перестала усваивать новые паттерны.")
            break
            
    # Формирование строки для итогового отчета
    model_summary = f"BN:{model_kwargs.get('use_bn', False)}, DP:{model_kwargs.get('use_dropout', False)}"
    results.append({
        'experiment_id': exp_id, 'dataset': 'EMNIST', 'seed': 42,
        'model_summary': model_summary, 'optimizer': opt_name, 'lr': lr,
        'momentum': momentum, 'weight_decay': weight_decay,
        'epochs_trained': epochs_trained, 'best_val_accuracy': best_val_acc, 'best_val_loss': best_val_loss
    })
    
    histories[exp_id] = history
    
    # Сохранение артефактов только для финальной модели E4
    if exp_id == 'E4' and best_model_state:
        torch.save(best_model_state, "artifacts/best_model.pt")
        with open("artifacts/best_config.json", "w") as f:
            json.dump({
                "dataset": "EMNIST", "seed": 42, "model_kwargs": model_kwargs,
                "optimizer": opt_name, "lr": lr, "early_stopping_patience": patience
            }, f, indent=4)
            
    return best_val_acc, model_kwargs

In [8]:
import pandas as pd
from IPython.display import display

# Часть A (S08): Регуляризация
print("Инициализация E1: Базовая архитектура без защиты...")
run_experiment('E1', {'use_bn': False, 'use_dropout': False}, 'Adam', 1e-3, epochs=10)

print("\nИнициализация E2: Внедрение хаоса (Dropout)...")
val_acc_e2, kwargs_e2 = run_experiment('E2', {'use_bn': False, 'use_dropout': True, 'drop_p': 0.3}, 'Adam', 1e-3, epochs=10)

print("\nИнициализация E3: Жесткий контроль дисперсии (BatchNorm)...")
val_acc_e3, kwargs_e3 = run_experiment('E3', {'use_bn': True, 'use_dropout': False}, 'Adam', 1e-3, epochs=10)

# Стратегический выбор лучшей защиты
best_kwargs = kwargs_e2 if val_acc_e2 > val_acc_e3 else kwargs_e3
print(f"\nАнализ завершен. Оптимальная конфигурация для E4: {best_kwargs}")

print("\nИнициализация E4: Лучшая модель + Механизм ранней остановки (EarlyStopping)...")
run_experiment('E4', best_kwargs, 'Adam', 1e-3, epochs=20, patience=3)

# Часть B (S09): Оптимизация и её деградация
print("\nИнициализация O1: Саботаж шага обучения (Огромный LR = 1e-1)...")
run_experiment('O1', best_kwargs, 'Adam', lr=1e-1, epochs=6)

print("\nИнициализация O2: Саботаж шага обучения (Микроскопический LR = 1e-5)...")
run_experiment('O2', best_kwargs, 'Adam', lr=1e-5, epochs=6)

print("\nИнициализация O3: Классический алгоритм (SGD + momentum + weight_decay)...")
run_experiment('O3', best_kwargs, 'SGD', lr=1e-2, momentum=0.9, weight_decay=1e-4, epochs=10)

# Сбор разведданных
df_runs = pd.DataFrame(results)
df_runs.to_csv("artifacts/runs.csv", index=False)
print("\nЦикл экспериментов завершен. Данные экспортированы в artifacts/runs.csv.")
display(df_runs)

Инициализация E1: Базовая архитектура без защиты...

Инициализация E2: Внедрение хаоса (Dropout)...

Инициализация E3: Жесткий контроль дисперсии (BatchNorm)...

Анализ завершен. Оптимальная конфигурация для E4: {'use_bn': True, 'use_dropout': False}

Инициализация E4: Лучшая модель + Механизм ранней остановки (EarlyStopping)...
[E4] Обучение принудительно прервано на эпохе 11. Модель перестала усваивать новые паттерны.

Инициализация O1: Саботаж шага обучения (Огромный LR = 1e-1)...

Инициализация O2: Саботаж шага обучения (Микроскопический LR = 1e-5)...

Инициализация O3: Классический алгоритм (SGD + momentum + weight_decay)...

Цикл экспериментов завершен. Данные экспортированы в artifacts/runs.csv.


,experiment_id,dataset,seed,model_summary,optimizer,lr,momentum,weight_decay,epochs_trained,best_val_accuracy,best_val_loss
0,E1,EMNIST,42,"BN:False, DP:False",Adam,0.00100,0.0,0.0000,10,0.832402,0.521391
1,E2,EMNIST,42,"BN:False, DP:True",Adam,0.00100,0.0,0.0000,10,0.809353,0.592088
2,E3,EMNIST,42,"BN:True, DP:False",Adam,0.00100,0.0,0.0000,10,0.852615,0.456821
3,E4,EMNIST,42,"BN:True, DP:False",Adam,0.00100,0.0,0.0000,11,0.852615,0.456821
4,O1,EMNIST,42,"BN:True, DP:False",Adam,0.10000,0.0,0.0000,6,0.793395,0.664938
5,O2,EMNIST,42,"BN:True, DP:False",Adam,0.00001,0.0,0.0000,6,0.685417,1.503271
6,O3,EMNIST,42,"BN:True, DP:False",SGD,0.01000,0.9,0.0001,10,0.852704,0.447544


In [9]:
import matplotlib.pyplot as plt

# 1. Визуализация лучшей модели (E4)
h_e4 = histories['E4']
epochs_e4 = range(1, len(h_e4['train_loss']) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_e4, h_e4['train_loss'], label='Train Loss', marker='o', color='darkred')
plt.plot(epochs_e4, h_e4['val_loss'], label='Val Loss', marker='o', color='black')
plt.title('E4: Loss Curve (EarlyStopping)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_e4, h_e4['train_acc'], label='Train Acc', marker='o', color='darkgreen')
plt.plot(epochs_e4, h_e4['val_acc'], label='Val Acc', marker='o', color='black')
plt.title('E4: Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("artifacts/figures/curves_best.png", bbox_inches='tight')
plt.close()

# 2. Визуализация деградации шага обучения (O1, O2)
plt.figure(figsize=(12, 5))

h_o1 = histories['O1']
epochs_o1 = range(1, len(h_o1['train_loss']) + 1)
plt.subplot(1, 2, 1)
plt.plot(epochs_o1, h_o1['train_loss'], label='Train Loss', color='red')
plt.plot(epochs_o1, h_o1['val_loss'], label='Val Loss', color='orange')
plt.title('O1: Саботаж (Огромный LR = 1e-1)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()

h_o2 = histories['O2']
epochs_o2 = range(1, len(h_o2['train_loss']) + 1)
plt.subplot(1, 2, 2)
plt.plot(epochs_o2, h_o2['train_loss'], label='Train Loss', color='blue')
plt.plot(epochs_o2, h_o2['val_loss'], label='Val Loss', color='cyan')
plt.title('O2: Стагнация (Микро LR = 1e-5)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("artifacts/figures/curves_lr_extremes.png", bbox_inches='tight')
plt.close()

# 3. Финальная беспристрастная оценка на тестовой выборке
best_model = MLP(**best_kwargs).to(device)
best_model.load_state_dict(torch.load("artifacts/best_model.pt", weights_only=True))
test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss())

print("-" * 50)
print(f"АБСОЛЮТНЫЙ ИТОГ (TEST DATA):")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print("-" * 50)

--------------------------------------------------
АБСОЛЮТНЫЙ ИТОГ (TEST DATA):
Loss: 0.4709
Accuracy: 0.8463
--------------------------------------------------
